In [5]:
!pip install CHAID

In [ ]:
import pandas as pd

# Kevin Hillstromのサイトから直接CSVを読み込む
url = "http://www.minethatdata.com/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
df = pd.read_csv(url)

# データが無事に取得できたか、先頭5行を表示して確認
display(df.head())

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [2]:
# 各セグメントの人数と割合を確認する
display(df['segment'].value_counts())

,count
segment,
Womens E-Mail,21387
Mens E-Mail,21307
No E-Mail,21306


In [3]:
# 前処理（特徴量エンジニアリング）
# CHAIDが扱いやすいように、データを少し整理します。

# ① 目的変数の設定：今回は「visit（サイト訪問）」ではなく、よりビジネスインパクトのある「conversion（購入）」をターゲットにします
target = 'conversion'

# ② 使用する説明変数の選択
# 今回の分析ストーリーに沿って、解釈しやすいカテゴリ変数を中心にピックアップします
features = [
    'recency',       # 最終購入からの月数（数値ですが、CHAIDは自動でカテゴリ化してくれます）
    'history_segment', # 過去の購買金額のランク（文字列）
    'zip_code',      # 居住地域（Urban, Suburban, Rural）
    'newbie',        # 新規顧客か否か（1 or 0）
    'channel',       # 過去の購買チャネル（Phone, Web, Multichannel）
    'segment'        # ★今回のアクション変数（Mens E-Mail, Womens E-Mail, No E-Mail）
]

# 分析用のデータフレームを作成
df_model = df[features + [target]].copy()

# newbie（新規顧客）フラグは数値（1/0）になっているため、分かりやすく文字列に変換しておきます
df_model['newbie'] = df_model['newbie'].map({1: '新規', 0: '既存'})

# 準備完了したデータの確認
display(df_model.head())
display(df_model.info())

,recency,history_segment,zip_code,newbie,channel,segment,conversion
0,10,2) $100 - $200,Surburban,既存,Phone,Womens E-Mail,0
1,6,3) $200 - $350,Rural,新規,Web,No E-Mail,0
2,7,2) $100 - $200,Surburban,新規,Web,Womens E-Mail,0
3,9,5) $500 - $750,Rural,新規,Web,Mens E-Mail,0
4,2,1) $0 - $100,Urban,既存,Web,Womens E-Mail,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   recency          64000 non-null  int64 
 1   history_segment  64000 non-null  object
 2   zip_code         64000 non-null  object
 3   newbie           64000 non-null  object
 4   channel          64000 non-null  object
 5   segment          64000 non-null  object
 6   conversion       64000 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 3.4+ MB


None

In [7]:
from CHAID import Tree

# CHAIDに各変数の「データの種類（尺度）」を教えてあげるための辞書を作成します
# nominal: 名義尺度（順序に意味がないカテゴリ。例：地域、性別）
# ordinal: 順序尺度（順序に意味があるもの。例：経過月数、ランク）
features_dict = {
    'recency': 'ordinal',        # 最終購入からの月数（大小に意味がある）
    'history_segment': 'nominal',# 過去の購買金額ランク
    'zip_code': 'nominal',       # 居住地域
    'newbie': 'nominal',         # 新規顧客フラグ
    'channel': 'nominal',        # 購買チャネル
    'segment': 'nominal'         # 配信したEメールの種類
}

# CHAIDモデルの構築
tree = Tree.from_pandas_df(
    df_model,
    i_variables=features_dict,       # ★リストではなく、作成した辞書を渡す
    d_variable=target,               # 目的変数（conversion）
    alpha_merge=0.05,                # カイ二乗検定の有意水準
    max_depth=3,                     # 木の深さ
    min_parent_node_size=500,        # 親ノードの最小サンプル数
    min_child_node_size=200          # 子ノードの最小サンプル数
)

# ツリー構造のコンソール出力
print("■ Kevin Hillstrom データセット：CHAID 分類ツリー")
tree.print_tree()

■ Kevin Hillstrom データセット：CHAID 分類ツリー
([], {0: 63422.0, 1: 578.0}, (history_segment, p=3.928007839599126e-13, score=60.819382601591485, groups=[['1) $0 - $100', '2) $100 - $200'], ['3) $200 - $350'], ['4) $350 - $500', '6) $750 - $1,000', '5) $500 - $750'], ['7) $1,000 +']]), dof=3))
|-- (['1) $0 - $100', '2) $100 - $200'], {0: 36961.0, 1: 263.0}, (segment, p=2.413508679021971e-06, score=22.234085913667425, groups=[['Mens E-Mail', 'Womens E-Mail'], ['No E-Mail']]), dof=1))
|   |-- (['Mens E-Mail', 'Womens E-Mail'], {0: 24565.0, 1: 211.0}, (recency, p=0.007568449062945799, score=7.132846325848508, groups=[[1], [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]]), dof=1))
|   |   |-- ([1], {0: 2474.0, 1: 33.0}, <Invalid Chaid Split> - the max depth has been reached)
|   |   +-- ([2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], {0: 22091.0, 1: 178.0}, <Invalid Chaid Split> - the max depth has been reached)
|   +-- (['No E-Mail'], {0: 12396.0, 1: 52.0}, (newbie, p=0.00024705436349124526, score=13.434381949452566, g

In [8]:
# Streamlitで使うためのデータをCSVとして保存
df_model.to_csv('kevin_hillstrom_for_app.csv', index=False)